In [2]:
import torch
import torchvision
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

%config InlineBackend.figure_format = 'svg'
plt.style.use('seaborn-v0_8')

In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [6]:


!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip

import pandas as pd

ratings_data = pd.read_csv('/content/ml-latest-small/ratings.csv')
movie_names_data = pd.read_csv('/content/ml-latest-small/movies.csv')
print(ratings_data.head())
print(movie_names_data.head())

--2026-04-29 16:25:49--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip.1’

ml-latest-small.zip 100%[===================>] 955.28K  1.03MB/s    in 0.9s    

2026-04-29 16:25:50 (1.03 MB/s) - ‘ml-latest-small.zip.1’ saved [978202/978202]

Archive:  ml-latest-small.zip
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  9649829

In [7]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())

In [8]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [9]:
ratings_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [10]:
from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [11]:
user_enc.classes_[4], movie_enc.classes_[8871]

(np.int64(5), 'Toy Story (1995)')

In [12]:
for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[np.int64(0), np.int64(8871)] 4.0
[np.int64(0), np.int64(3661)] 4.0
[np.int64(0), np.int64(3845)] 4.0
[np.int64(0), np.int64(7523)] 5.0
[np.int64(0), np.int64(9119)] 5.0
[np.int64(0), np.int64(3252)] 3.0
[np.int64(0), np.int64(1284)] 5.0
[np.int64(0), np.int64(1337)] 4.0
[np.int64(0), np.int64(7180)] 5.0
[np.int64(0), np.int64(1535)] 5.0


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [15]:
num_users = len(X)
num_movies = len(X)

In [16]:
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

def build_model():
    movie_input = Input(shape=[1], name="Book-Input")
    movie_embedding = Embedding(n_movies+1, 15, name="Book-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Books")(movie_embedding)

    user_input = Input(shape=[1], name="User-Input")
    user_embedding = Embedding(n_user+1, 15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)

    prod = Dot(name="Dot-Product", axes=1)([user_vec, movie_vec])

    prod = Dense(32)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)
    prod = Dense(1)(prod)


    model = Model([user_input, movie_input], prod)
    model.compile('adam', 'mean_squared_error', metrics=['accuracy'])

    return model


model = build_model()

In [18]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    [X_train[:, 0], X_train[:, 1]],
    Y_train,
    epochs=15,
    verbose=1,
    batch_size=64,
    validation_data=([X_test[:, 0], X_test[:, 1]], Y_test),
    callbacks=[model_checkpoint_callback]
)

Epoch 1/15
1243/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0232 - loss: 4.3611
Epoch 1: val_loss improved from None to 1.15787, saving model to ./checkpoint.weights.h5

Epoch 1: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.0271 - loss: 2.7039 - val_accuracy: 0.0280 - val_loss: 1.1579
Epoch 2/15
1260/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0275 - loss: 1.7290
Epoch 2: val_loss improved from 1.15787 to 0.97229, saving model to ./checkpoint.weights.h5

Epoch 2: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0279 - loss: 1.6123 - val_accuracy: 0.0280 - val_loss: 0.9723
Epoch 3/15
1249/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0284 - loss: 1.1990
Epoch 3: val_loss improved from 0.97229 to 0.92263, saving model to ./checkpoint.weights.h5

Epoch 3: finished saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - acc

In [19]:
X_test[:5], Y_test[:5]

(array([[ 275, 4337],
        [ 598, 7425],
        [ 482,  334],
        [ 201, 3548],
        [ 273, 3540]]),
 41008    5.0
 94274    2.5
 77380    2.5
 29744    3.0
 40462    4.0
 Name: rating, dtype: float64)

In [20]:
predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step


In [21]:
print(predictions,"\n\n", Y_test[:5].values)

[[3.9759767]
 [2.963325 ]
 [2.9161196]
 [3.4036648]
 [3.0791419]] 

 [5.  2.5 2.5 3.  4. ]


In [22]:
movie_enc.classes_[4]

"'Til There Was You (1997)"

In [29]:
def extract_true_ratings(user_id, X_test, Y_test):
    user_indices = X_test[:, 0] == user_id
    movies = X_test[user_indices][:, 1]
    ratings = Y_test[user_indices]

    for movie, rating in zip(movies, ratings):
        print(f"Movie ID: {movie}, True Rating: {rating}")

In [30]:
user_id = 10

In [31]:
extract_true_ratings(user_id, X_test, Y_test)

Movie ID: 8644, True Rating: 4.0
Movie ID: 1403, True Rating: 3.0
Movie ID: 8363, True Rating: 4.0
Movie ID: 6529, True Rating: 3.0
Movie ID: 9298, True Rating: 3.0
Movie ID: 5754, True Rating: 4.0
Movie ID: 2150, True Rating: 3.0
Movie ID: 4309, True Rating: 4.0
Movie ID: 7165, True Rating: 2.0
Movie ID: 7680, True Rating: 5.0
Movie ID: 285, True Rating: 4.0
Movie ID: 1344, True Rating: 4.0
Movie ID: 3555, True Rating: 4.0


In [32]:
def extract_true_ratings(user_id, X_test):

    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [33]:
def predict_ratings(user_id, X_test):
    '''
    given user id predict all ratings for movies
    '''
    user_data = ratings_data[ratings_data['userId'] == user_id]
    movie_ids, movie_names, predictions, movie_genres = list(), list(), list(), list()
    i = 0
    for _id, movie_id in X_test:
        if user_id == X_test[i][0]:
            movie_ids.append(X_test[i, 1])
            movie_names.append(movie_enc.classes_[movie_id])
            pred = model.predict([ np.array([X_test[i, 0]]), np.array([X_test[i, 1]]) ])
            predictions.append(pred[0][0])
        i += 1
    return movie_ids, movie_names, movie_genres, predictions

In [34]:
test_user_id = 7
userid_rating_data = ratings_data[ratings_data['userId'] == test_user_id]

In [35]:
movie_ids, movie_names, movie_genres, predictions = predict_ratings(test_user_id, X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


In [36]:
dictionary = {"user_id": [test_user_id]*len(movie_ids),
              "movie_id": movie_ids,
              "movie_name":movie_names,
              "predicted_ratings":predictions,
              "true_ratings": extract_true_ratings(test_user_id, X_test)
              }

In [37]:
prediction_dataframe = pd.DataFrame.from_dict(dictionary, orient='index').transpose()
prediction_dataframe.sort_values('predicted_ratings', ascending=False)

,user_id,movie_id,movie_name,predicted_ratings,true_ratings
5,7,7421,Schindler's List (1993),4.482104,5.0
0,7,6865,Pulp Fiction (1994),3.398406,4.0
4,7,2139,Dances with Wolves (1990),3.311508,5.0
2,7,7912,Speed (1994),2.919593,4.0
3,7,1799,City Slickers II: The Legend of Curly's Gold (...,2.805202,1.0
1,7,7755,Sleepless in Seattle (1993),2.332388,3.0
